# PCA of top-5 overall transformed force-constant solutions

This notebook performs PCA on a five-dimensional transformed BvK force-constant vector
\[
(\alpha_0,\; \alpha_1+2\beta_1,\; \alpha_1-\beta_1,\; \alpha_2,\; \beta_2).
\]

The original first-neighbor parameters are still read from the dataframe, but in the PCA feature matrix:

- `alpha1` is replaced by `alpha1_plus_2beta1`
- `beta1` is replaced by `alpha1_minus_beta1`

The analysis uses the top five overall solutions from `dataframe000013.pkl`, `dataframe000014.pkl`, `dataframe000015.pkl`, and `dataframe000016.pkl` for each \((m,a)\) pair.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler

%config InlineBackend.figure_format = "retina"

DATA_DIR = Path("./dataframes/")  # Update if needed.

DATAFRAME_FILES = {
    "dataframe000013": DATA_DIR / "dataframe000013.pkl",
    "dataframe000014": DATA_DIR / "dataframe000014.pkl",
    "dataframe000015": DATA_DIR / "dataframe000015.pkl",
    "dataframe000016": DATA_DIR / "dataframe000016.pkl",
}

TOP_K = 5
RANK_BY = "fitness_norm"

SAVE_FORMATS = ("pdf", "png")
DPI = 300

FIGSIZE = (6.2, 5.2)
MARKER_SIZE = 58
MARKER_ALPHA = 0.86
EDGE_WIDTH = 0.3
ZERO_TOL = 0.08


In [ ]:
def find_first_existing(candidates, columns):
    for col in candidates:
        if col in columns:
            return col
    return None


def load_dataframe(path, dataset_label):
    if not path.exists():
        raise FileNotFoundError(f"Could not find {path}. Update DATA_DIR or DATAFRAME_FILES.")
    df = pd.read_pickle(path).copy()
    df["dataset"] = dataset_label
    return df


def display_label(col):
    label_map = {
        "mass": r"$m$ (amu)", "m": r"$m$ (amu)",
        "a_val": r"$a$ ($\AA$)", "alat": r"$a$ ($\AA$)",
        "a_latt": r"$a$ ($\AA$)", "lattice_parameter": r"$a$ ($\AA$)",
        "alpha0": r"$\alpha_0$ (N/m)", "alpha_0": r"$\alpha_0$ (N/m)",
        "alpha1": r"$\alpha_1$ (N/m)", "alpha_1": r"$\alpha_1$ (N/m)",
        "beta1": r"$\beta_1$ (N/m)", "beta_1": r"$\beta_1$ (N/m)",
        "alpha1_plus_2beta1": r"$\alpha_1 + 2\beta_1$ (N/m)",
        "alpha1_minus_beta1": r"$\alpha_1 - \beta_1$ (N/m)",
        "alpha2": r"$\alpha_2$ (N/m)", "alpha_2": r"$\alpha_2$ (N/m)",
        "beta2": r"$\beta_2$ (N/m)", "beta_2": r"$\beta_2$ (N/m)",
        "fitness_norm": r"$f_{\mathrm{norm}}$", "fnorm": r"$f_{\mathrm{norm}}$",
        "r_alpha2": r"$r_{\alpha_2}$",
    }
    return label_map.get(col, col)


def apply_publication_axes(ax):
    ax.tick_params(axis="both", which="both", direction="in", top=True, right=True,
                   labelsize=12, length=5, width=1.1)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.1)


def save_figure(fig, stem, output_dir, formats=SAVE_FORMATS):
    output_dir.mkdir(parents=True, exist_ok=True)
    saved = []
    for ext in formats:
        outpath = output_dir / f"{stem}.{ext}"
        fig.savefig(outpath, dpi=DPI, bbox_inches="tight")
        saved.append(outpath)
    print("Saved:")
    for path in saved:
        print(f"  {path}")
    return saved


def classify_family(row, alpha2_col, beta2_col):
    a2 = row[alpha2_col]
    b2 = row[beta2_col]
    if abs(b2) <= ZERO_TOL and abs(a2) > ZERO_TOL:
        return r"$\alpha_2$-dominated"
    if abs(a2) <= ZERO_TOL and abs(b2) > ZERO_TOL:
        return r"$\beta_2$-dominated"
    if abs(a2) > abs(b2):
        return r"$\alpha_2$ larger"
    if abs(b2) > abs(a2):
        return r"$\beta_2$ larger"
    return "balanced"


In [ ]:
frames = []
for label, path in DATAFRAME_FILES.items():
    tmp = load_dataframe(path, label)
    print(f"{label}: {tmp.shape[0]:,} rows, {tmp.shape[1]:,} columns")
    frames.append(tmp)

df_all = pd.concat(frames, ignore_index=True)
print(f"\nAggregated dataframe before filtering: {df_all.shape[0]:,} rows")

mass_col = find_first_existing(["mass", "m"], df_all.columns)
alat_col = find_first_existing(["a_val", "alat", "a_latt", "lattice_parameter"], df_all.columns)
rank_col = find_first_existing([RANK_BY, "fitness_norm", "fnorm"], df_all.columns)

alpha0_col = find_first_existing(["alpha0", "alpha_0"], df_all.columns)
alpha1_col = find_first_existing(["alpha1", "alpha_1"], df_all.columns)
beta1_col  = find_first_existing(["beta1", "beta_1"], df_all.columns)
alpha2_col = find_first_existing(["alpha2", "alpha_2"], df_all.columns)
beta2_col  = find_first_existing(["beta2", "beta_2"], df_all.columns)

original_force_constant_cols = [alpha0_col, alpha1_col, beta1_col, alpha2_col, beta2_col]
required = [mass_col, alat_col, rank_col] + original_force_constant_cols
if any(col is None for col in required):
    raise ValueError("Could not detect all required columns. Check dataframe column names.")

print("\nOriginal force-constant columns detected:")
for col in original_force_constant_cols:
    print(f"  {col}")

df_top = (
    df_all.sort_values(rank_col, ascending=False)
    .groupby(["dataset", mass_col, alat_col], group_keys=False)
    .head(TOP_K)
    .copy()
)

for col in [mass_col, alat_col, rank_col] + original_force_constant_cols:
    df_top[col] = pd.to_numeric(df_top[col], errors="coerce")

df_top = df_top.dropna(subset=[mass_col, alat_col, rank_col] + original_force_constant_cols).copy()

# -------------------------------------------------------------------
# Transformed first-neighbor coordinates used for PCA
# -------------------------------------------------------------------
df_top["alpha1_plus_2beta1"] = df_top[alpha1_col] + 2.0 * df_top[beta1_col]
df_top["alpha1_minus_beta1"] = df_top[alpha1_col] - df_top[beta1_col]

force_constant_cols = [
    alpha0_col,
    "alpha1_plus_2beta1",
    "alpha1_minus_beta1",
    alpha2_col,
    beta2_col,
]

print("\nPCA feature columns:")
for col in force_constant_cols:
    print(f"  {col}: {display_label(col)}")

df_top["abs_alpha2"] = df_top[alpha2_col].abs()
df_top["abs_beta2"] = df_top[beta2_col].abs()
df_top["r_alpha2"] = df_top["abs_alpha2"] / (df_top["abs_alpha2"] + df_top["abs_beta2"] + 1e-12)
df_top["delta_alpha2_beta2"] = df_top["abs_alpha2"] - df_top["abs_beta2"]
df_top["family"] = df_top.apply(lambda row: classify_family(row, alpha2_col, beta2_col), axis=1)

print(f"\nRows after top-{TOP_K}-overall selection: {df_top.shape[0]:,}")
print(df_top["family"].value_counts())

X = df_top[force_constant_cols].to_numpy(dtype=float)
X_scaled = StandardScaler().fit_transform(X)


In [ ]:
from sklearn.decomposition import PCA

OUTPUT_DIR = Path("pca_force_constant_space_top5_overall_transformed_alpha1_beta1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pca = PCA(n_components=5)
scores = pca.fit_transform(X_scaled)

for i in range(scores.shape[1]):
    df_top[f"PC{i+1}"] = scores[:, i]

explained = pca.explained_variance_ratio_
print("Explained variance ratio:")
for i, val in enumerate(explained, start=1):
    print(f"PC{i}: {val:.4f}")

loadings = pd.DataFrame(
    pca.components_.T,
    index=[display_label(c) for c in force_constant_cols],
    columns=[f"PC{i}" for i in range(1, 6)],
)
loadings


In [ ]:
def plot_pca_scatter(color_col, cmap, stem, color_label=None):
    fig, ax = plt.subplots(figsize=FIGSIZE)
    c_all = df_top[color_col].astype(float)
    markers = {"dataframe000013": "o", "dataframe000014": "s", "dataframe000015": "^", "dataframe000016": "D"}
    scatter_ref = None

    for dataset_label, sub in df_top.groupby("dataset"):
        sc = ax.scatter(
            sub["PC1"], sub["PC2"],
            c=sub[color_col].astype(float),
            cmap=cmap, vmin=c_all.min(), vmax=c_all.max(),
            s=MARKER_SIZE, alpha=MARKER_ALPHA,
            edgecolors="black", linewidths=EDGE_WIDTH,
            marker=markers.get(dataset_label, "o"),
            label=dataset_label,
        )
        scatter_ref = sc

    ax.set_xlabel(f"PC1 ({explained[0]*100:.1f}%)", fontsize=14)
    ax.set_ylabel(f"PC2 ({explained[1]*100:.1f}%)", fontsize=14)
    ax.set_title("PCA of transformed BvK force-constant space", fontsize=15, pad=10)
    apply_publication_axes(ax)
    ax.legend(frameon=False, fontsize=9, loc="best")

    cbar = fig.colorbar(scatter_ref, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label(color_label or display_label(color_col), fontsize=13)
    cbar.ax.tick_params(labelsize=11)

    fig.tight_layout()
    save_figure(fig, stem, OUTPUT_DIR)
    plt.show()

plot_pca_scatter(alat_col, "viridis", "pca_pc1_pc2_colored_by_lattice_parameter", r"$a$ ($\AA$)")
plot_pca_scatter(mass_col, "plasma", "pca_pc1_pc2_colored_by_mass", r"$m$ (amu)")
plot_pca_scatter("r_alpha2", "coolwarm", "pca_pc1_pc2_colored_by_r_alpha2", r"$r_{\alpha_2}$")


In [ ]:
fig, ax = plt.subplots(figsize=FIGSIZE)

for family, sub in df_top.groupby("family"):
    ax.scatter(
        sub["PC1"], sub["PC2"],
        s=MARKER_SIZE, alpha=MARKER_ALPHA,
        edgecolors="black", linewidths=EDGE_WIDTH,
        label=family,
    )

ax.set_xlabel(f"PC1 ({explained[0]*100:.1f}%)", fontsize=14)
ax.set_ylabel(f"PC2 ({explained[1]*100:.1f}%)", fontsize=14)
ax.set_title("Transformed PCA colored by force-constant family", fontsize=15, pad=10)
apply_publication_axes(ax)
ax.legend(frameon=False, fontsize=9, loc="best")
fig.tight_layout()
save_figure(fig, "pca_pc1_pc2_colored_by_family", OUTPUT_DIR)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(5.4, 4.2))
pcs = np.arange(1, len(explained) + 1)
ax.plot(pcs, explained * 100, marker="o", linewidth=1.5)
ax.set_xlabel("Principal component", fontsize=14)
ax.set_ylabel("Explained variance (%)", fontsize=14)
ax.set_title("PCA explained variance", fontsize=15, pad=10)
ax.set_xticks(pcs)
apply_publication_axes(ax)
fig.tight_layout()
save_figure(fig, "pca_scree_plot", OUTPUT_DIR)
plt.show()

loadings.to_csv(OUTPUT_DIR / "pca_loadings.csv")
df_top.to_csv(OUTPUT_DIR / "pca_scores_top5_overall.csv", index=False)


In [ ]:
# ============================================================
# PCA BIPLOT
# ============================================================

feature_names = [
    r"$\alpha_0$",
    r"$\alpha_1 + 2\beta_1$",
    r"$\alpha_1 - \beta_1$",
    r"$\alpha_2$",
    r"$\beta_2$",
]

loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f"PC{i+1}" for i in range(5)],
    index=feature_names,
)

fig, ax = plt.subplots(figsize=(8, 6))

# PCA scores

ax.scatter(
    scores[:, 0],
    scores[:, 1],
    s=70,
    alpha=0.7,
    edgecolor="black",
    linewidth=0.5,
)

# Loading vectors

scale = 3.0

for feature in feature_names:

    x = loadings.loc[feature, "PC1"] * scale
    y = loadings.loc[feature, "PC2"] * scale

    ax.arrow(
        0,
        0,
        x,
        y,
        head_width=0.08,
        head_length=0.12,
        linewidth=2,
        length_includes_head=True,
    )

    ax.text(
        x * 1.15,
        y * 1.15,
        feature,
        fontsize=16,
        ha="center",
        va="center",
    )

pc1_var = 100 * pca.explained_variance_ratio_[0]
pc2_var = 100 * pca.explained_variance_ratio_[1]

ax.set_xlabel(f"PC1 ({pc1_var:.1f}%)")
ax.set_ylabel(f"PC2 ({pc2_var:.1f}%)")
ax.set_title("PCA biplot of transformed BvK force-constant space")

ax.axhline(0, color="0.8")
ax.axvline(0, color="0.8")

ax.tick_params(
    axis='both',
    which='both',
    direction='in',
    top=True,
    right=True,
    labelsize=16,
    length=6,
    width=1.2,
)

for spine in ax.spines.values():
    spine.set_linewidth(1.2)

fig.tight_layout()

plt.savefig(
    OUTPUT_DIR / "pca_biplot_transformed_force_constants.pdf",
    bbox_inches="tight"
)

plt.show()